# Trial Analysis Notebook

This notebook performs a multi-phase analysis of trial data from a raw events CSV.

- **Phase 1:** Parses the raw CSV to identify the precise start, end, and intermediate marked events for each trial. The output is a structured **event timeline**.
- **Phase 2:** Parses the same raw CSV to identify all mode-changing commands and build a continuous **mode timeline** for the entire recording.

## Setup: Imports, Configuration, and Helper Functions

In [20]:
import json
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
from collections import defaultdict

# --- Variables to Configure ---
cohort_number = 4
input_csv_file = Path(f'../../data/cohort-{cohort_number}-raw.csv')
output_event_timeline_file = Path(f'../../data/cohort-{cohort_number}-event-timeline.yaml')
output_mode_timeline_file = Path(f'../../data/cohort-{cohort_number}-mode-timeline.yaml') # New output file
final_output_file = Path(f'../../data/cohort-{cohort_number}-summary.csv')

# --- Constants ---
# Time threshold to group button press events.
EVENT_GROUPING_THRESHOLD_NS = 75_000_000  # 75 milliseconds

# --- Helper Functions ---
def robust_csv_value_to_bool(value) -> bool:
    """Converts a value read from a CSV cell to a boolean robustly."""
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return bool(value)
    return str(value).strip().lower() == 'true'

def process_and_finalize_event_group(group_ts_list: list) -> int:
    """Calculates the average timestamp of an event group."""
    if not group_ts_list:
        return 0
    return (group_ts_list[0] + group_ts_list[-1]) // 2

---

## Phase 1: Create Trial Event Timeline

In [4]:
def create_event_timeline(df: pd.DataFrame):
    """Parses the DataFrame to find trial start, end, and marked events."""
    # Ensure required columns are present
    required_events = ['researcher-trial-start', 'researcher-trial-end', 'researcher-mark-event']
    for col in required_events:
        if col not in df.columns:
            raise ValueError(f"Error: Required column '{col}' not found.")
        df[col] = df[col].apply(robust_csv_value_to_bool)
        
    event_timeline = {}
    current_trial_number = 1
    active_trial_start_ts, last_start_signal_ts = None, None
    active_trial_marked_events, current_marked_event_group_ts = [], []

    print("Phase 1: Parsing CSV to build event timeline...")
    for index, row in df.iterrows():
        timestamp_ns = row['timestamp']
        
        # Trial Start Logic (on button release)
        if row['researcher-trial-start']:
            if active_trial_start_ts: print(f"Warning (row {index + 2}): New 'start' signal while trial {current_trial_number} was active. Aborting.")
            last_start_signal_ts = timestamp_ns
            active_trial_start_ts = None

        elif last_start_signal_ts and not active_trial_start_ts:
            active_trial_start_ts = last_start_signal_ts
            print(f"TRIAL {current_trial_number} START detected at timestamp: {active_trial_start_ts}")
            active_trial_marked_events, current_marked_event_group_ts = [], []
        
        # Logic during an active trial
        if active_trial_start_ts:
            # Marked Event Logic
            if row['researcher-mark-event']:
                if current_marked_event_group_ts and (timestamp_ns - current_marked_event_group_ts[-1]) > EVENT_GROUPING_THRESHOLD_NS:
                    final_event_ts = process_and_finalize_event_group(current_marked_event_group_ts)
                    active_trial_marked_events.append(final_event_ts)
                    current_marked_event_group_ts = [timestamp_ns]
                else:
                    current_marked_event_group_ts.append(timestamp_ns)
            
            # Trial End Logic
            if row['researcher-trial-end']:
                trial_end_ts = timestamp_ns
                print(f"TRIAL {current_trial_number} END detected at timestamp: {trial_end_ts}")
                if current_marked_event_group_ts:
                    final_event_ts = process_and_finalize_event_group(current_marked_event_group_ts)
                    active_trial_marked_events.append(final_event_ts)
                
                trial_data = {'trial-start': {'timestamp': active_trial_start_ts, 'trial-rel-time': 0.0}}
                for i, event_ts in enumerate(sorted(active_trial_marked_events)):
                    rel_time = (event_ts - active_trial_start_ts) / 1e9
                    trial_data[f'event-{i+1}'] = {'timestamp': event_ts, 'trial-rel-time': round(rel_time, 4)}
                
                rel_time = (trial_end_ts - active_trial_start_ts) / 1e9
                trial_data['trial-end'] = {'timestamp': trial_end_ts, 'trial-rel-time': round(rel_time, 4)}
                event_timeline[str(current_trial_number)] = trial_data
                
                current_trial_number += 1
                active_trial_start_ts, last_start_signal_ts = None, None
                active_trial_marked_events, current_marked_event_group_ts = [], []
    
    print("\nPhase 1 parsing complete.")
    return event_timeline

In [5]:
# Load the main CSV
try:
    main_df = pd.read_csv(input_csv_file, keep_default_na=True, na_values=[''])
    print(f"Successfully loaded {input_csv_file}\n")
    
    # Execute Phase 1
    event_timeline_result = create_event_timeline(main_df.copy()) # Use copy to avoid side-effects
    
    # Save and inspect the result
    if event_timeline_result:
        with open(output_event_timeline_file, 'w') as f:
            yaml.dump(event_timeline_result, f, indent=2, default_flow_style=False, sort_keys=False)
        print(f"\nEvent timeline saved to: {output_event_timeline_file}")
    else:
        print("No complete trials were found. No event timeline generated.")

except FileNotFoundError:
    print(f"FATAL ERROR: Input CSV file not found at '{input_csv_file}'")


Successfully loaded ../../data/cohort-1-raw.csv

Phase 1: Parsing CSV to build event timeline...
TRIAL 1 START detected at timestamp: 1750699384928464949
Warning (row 28121): New 'start' signal while trial 1 was active. Aborting.
TRIAL 1 START detected at timestamp: 1750699384978618408
Warning (row 28129): New 'start' signal while trial 1 was active. Aborting.
TRIAL 1 START detected at timestamp: 1750699385028798523
Warning (row 28138): New 'start' signal while trial 1 was active. Aborting.
TRIAL 1 START detected at timestamp: 1750699385078879512
Warning (row 28146): New 'start' signal while trial 1 was active. Aborting.
TRIAL 1 START detected at timestamp: 1750699385129179952
Warning (row 28155): New 'start' signal while trial 1 was active. Aborting.
TRIAL 1 START detected at timestamp: 1750699385179256276
Warning (row 28163): New 'start' signal while trial 1 was active. Aborting.
TRIAL 1 START detected at timestamp: 1750699385229488510
Warning (row 28172): New 'start' signal while tr

---

## Phase 2: Create Continuous Mode Timeline

In [6]:
# Configuration for all mode-related events, including their resulting mode and precedence.
# Lower precedence number means higher priority.
MODE_EVENT_CONFIG = {
    # Stop Events (Highest Precedence)
    "supervisor-stop":           {"mode": "stop",     "precedence": 1},
    "teammate-stop":             {"mode": "stop",     "precedence": 1},
    "operator-stop":             {"mode": "stop",     "precedence": 1},
    # Operator Events (Medium Precedence)
    "researcher-enable-operator": {"mode": "operator", "precedence": 2},
    "supervisor-enable-operator": {"mode": "operator", "precedence": 2},
    "operator-enable-operator":  {"mode": "operator", "precedence": 2},
    # Autonomy Events (Lowest Precedence)
    "researcher-enable-autonomy": {"mode": "autonomy", "precedence": 3},
    "supervisor-enable-autonomy": {"mode": "autonomy", "precedence": 3}
}

In [7]:
def create_mode_timeline(df: pd.DataFrame):
    """Parses the DataFrame to build a continuous timeline of system modes."""
    mode_event_names = list(MODE_EVENT_CONFIG.keys())
    
    # Ensure required columns are present and boolean
    for col in mode_event_names:
        if col not in df.columns:
            raise ValueError(f"Error: Required mode event column '{col}' not found.")
        df[col] = df[col].apply(robust_csv_value_to_bool)
    
    # 1. Find all raw button press groups for mode-changing events
    print("Phase 2: Finding all mode event groups...")
    finalized_events = []
    for event_name in mode_event_names:
        current_event_group_ts = []
        for index, row in df.iterrows():
            if row[event_name]:
                timestamp_ns = row['timestamp']
                if current_event_group_ts and (timestamp_ns - current_event_group_ts[-1]) > EVENT_GROUPING_THRESHOLD_NS:
                    final_ts = process_and_finalize_event_group(current_event_group_ts)
                    finalized_events.append({'timestamp': final_ts, 'event': event_name})
                    current_event_group_ts = [timestamp_ns]
                else:
                    current_event_group_ts.append(timestamp_ns)
        if current_event_group_ts: # Finalize any remaining group
            final_ts = process_and_finalize_event_group(current_event_group_ts)
            finalized_events.append({'timestamp': final_ts, 'event': event_name})
    
    if not finalized_events:
        print("No mode-changing events found in file.")
        return {}
    
    # 2. Sort all events by time and resolve precedence for simultaneous events
    print("Resolving event precedence...")
    events_df = pd.DataFrame(finalized_events).sort_values('timestamp').reset_index(drop=True)
    events_df['precedence'] = events_df['event'].apply(lambda x: MODE_EVENT_CONFIG[x]['precedence'])
    
    # Group by timestamp, find the index of the minimum precedence for each group, and select that row
    final_transitions_df = events_df.loc[events_df.groupby('timestamp')['precedence'].idxmin()]

    # 3. Build the final mode timeline dictionary
    print("Building final mode timeline...")
    mode_timeline = {}
    # Add the initial state
    first_timestamp = df['timestamp'].iloc[0]
    mode_timeline[first_timestamp] = {'event': 'bag-start', 'mode': 'unknown'}
    
    for index, row in final_transitions_df.iterrows():
        event_name = row['event']
        timestamp = row['timestamp']
        mode = MODE_EVENT_CONFIG[event_name]['mode']
        mode_timeline[timestamp] = {'event': event_name, 'mode': mode}
        
    print("\nPhase 2 parsing complete.")
    return mode_timeline

In [8]:
# Execute Phase 2
try:
    mode_timeline_result = create_mode_timeline(main_df.copy()) # Use a copy
    
    # Save and inspect the result
    if mode_timeline_result:
        with open(output_mode_timeline_file, 'w') as f:
            # Use a custom representer to handle int keys gracefully in YAML
            yaml.add_representer(int, lambda d, data: yaml.ScalarNode('tag:yaml.org,2002:int', str(data)))
            yaml.dump(mode_timeline_result, f, indent=2, default_flow_style=False)
        print(f"\nMode timeline saved to: {output_mode_timeline_file}")
    else:
        print("No mode events were found. No mode timeline generated.")
        
except NameError:
    print("Could not execute Phase 2. Was the main DataFrame loaded correctly in Phase 1?")
except Exception as e:
    print(f"An error occurred during Phase 2: {e}")

Phase 2: Finding all mode event groups...
Resolving event precedence...
Building final mode timeline...

Phase 2 parsing complete.

Mode timeline saved to: ../../data/cohort-1-mode-timeline.yaml


# Final Analysis

In [14]:
# --- Phase 3 (Simplified): Calculate Durations, Operator Mode Time, and Stop Counts ---

print("--- Starting Simplified Phase 3: Final Analysis ---")

# --- 1. Load the processed timeline files ---
try:
    with open(output_event_timeline_file, 'r') as f:
        event_timeline = yaml.safe_load(f)
    print(f"Successfully loaded event timeline from: {output_event_timeline_file}")
except FileNotFoundError:
    print(f"FATAL: Event timeline file not found. Please run Phase 1 first.")
    event_timeline = None
except Exception as e:
    print(f"Error loading event timeline YAML: {e}")
    event_timeline = None

try:
    with open(output_mode_timeline_file, 'r') as f:
        mode_timeline_raw = yaml.safe_load(f)
    print(f"Successfully loaded mode timeline from: {output_mode_timeline_file}")
except FileNotFoundError:
    print(f"FATAL: Mode timeline file not found. Please run Phase 2 first.")
    mode_timeline_raw = None
except Exception as e:
    print(f"Error loading mode timeline YAML: {e}")
    mode_timeline_raw = None

# --- 2. Define the Core Analysis Function ---

def calculate_period_stats(start_ts: int, end_ts: int, mode_timeline: list):
    """
    Calculates operator mode duration and specific stop counts for a given time period.
    
    Args:
        start_ts: The start timestamp (ns) of the period.
        end_ts: The end timestamp (ns) of the period.
        mode_timeline: The full, sorted list of (timestamp, mode, event) tuples.
    
    Returns:
        A dictionary of calculated statistics for the period.
    """
    stats = {
        'operator_time_ns': 0,
        'operator_stop_count': 0,
        'teammate_stop_count': 0,
        'supervisor_stop_count': 0
    }
    
    period_duration_ns = end_ts - start_ts
    if period_duration_ns <= 0:
        return stats # Return zeroed stats for empty periods

    # Find the initial mode at the start of the period
    last_mode = 'unknown'
    for ts, mode, event in reversed(mode_timeline):
        if ts <= start_ts:
            last_mode = mode
            break
            
    last_ts = start_ts
    
    # Iterate through transitions that fall within the period
    for ts, mode, event in mode_timeline:
        if start_ts < ts <= end_ts:
            # Add duration for the mode that was active *before* this transition
            duration = ts - last_ts
            if last_mode == 'operator':
                stats['operator_time_ns'] += duration
            
            # Check for specific stop commands at the moment of transition
            if event == 'operator-stop': stats['operator_stop_count'] += 1
            if event == 'teammate-stop': stats['teammate_stop_count'] += 1
            if event == 'supervisor-stop': stats['supervisor_stop_count'] += 1
            
            # Update state for the next interval
            last_ts = ts
            last_mode = mode

    # Add duration for the final mode active until the end of the period
    if last_mode == 'operator':
        stats['operator_time_ns'] += end_ts - last_ts
    
    return stats

# --- 3. Process Timelines and Generate Final Report ---
if event_timeline and mode_timeline_raw:
    # Prepare the mode timeline for efficient lookup
    mode_timeline_sorted = sorted(
        [(ts, data['mode'], data['event']) for ts, data in mode_timeline_raw.items()]
    )
    
    final_results = []
    print("\nProcessing trials...")

    # Iterate through each trial from the event timeline
    for trial_num_str, trial_data in event_timeline.items():
        trial_num = int(trial_num_str)
        print(f" - Analyzing Trial {trial_num}...")
        
        # --- Get Trial Boundaries and Segments ---
        trial_start_ts = trial_data['trial-start']['timestamp']
        trial_end_ts = trial_data['trial-end']['timestamp']
        trial_duration_s = (trial_end_ts - trial_start_ts) / 1e9

        marked_events = [data for key, data in trial_data.items() if key.startswith('event-')]
        
        output_row = {'trial_number': trial_num}
        
        # --- Calculate Stats for the Entire Trial ---
        trial_stats = calculate_period_stats(trial_start_ts, trial_end_ts, mode_timeline_sorted)
        output_row['trial_duration_s'] = trial_duration_s
        output_row['trial_operator_time_s'] = trial_stats['operator_time_ns'] / 1e9
        output_row['trial_pct_time_operator'] = (output_row['trial_operator_time_s'] / trial_duration_s) * 100 if trial_duration_s > 0 else 0
        output_row['trial_operator_stop_count'] = trial_stats['operator_stop_count']
        output_row['trial_teammate_stop_count'] = trial_stats['teammate_stop_count']
        output_row['trial_supervisor_stop_count'] = trial_stats['supervisor_stop_count']
        
        # --- Calculate Stats for Segments ---
        if len(marked_events) == 2:
            output_row['comment'] = "OK"
            me1_ts = marked_events[0]['timestamp']
            me2_ts = marked_events[1]['timestamp']
            
            # Segment 1
            seg1_duration_s = (me1_ts - trial_start_ts) / 1e9
            seg1_stats = calculate_period_stats(trial_start_ts, me1_ts, mode_timeline_sorted)
            output_row['seg1_duration_s'] = seg1_duration_s
            output_row['seg1_operator_time_s'] = seg1_stats['operator_time_ns'] / 1e9
            output_row['seg1_pct_time_operator'] = (output_row['seg1_operator_time_s'] / seg1_duration_s) * 100 if seg1_duration_s > 0 else 0
            output_row['seg1_operator_stop_count'] = seg1_stats['operator_stop_count']
            output_row['seg1_teammate_stop_count'] = seg1_stats['teammate_stop_count']
            output_row['seg1_supervisor_stop_count'] = seg1_stats['supervisor_stop_count']
            
            # Segment 2
            seg2_duration_s = (me2_ts - me1_ts) / 1e9
            seg2_stats = calculate_period_stats(me1_ts, me2_ts, mode_timeline_sorted)
            output_row['seg2_duration_s'] = seg2_duration_s
            output_row['seg2_operator_time_s'] = seg2_stats['operator_time_ns'] / 1e9
            output_row['seg2_pct_time_operator'] = (output_row['seg2_operator_time_s'] / seg2_duration_s) * 100 if seg2_duration_s > 0 else 0
            output_row['seg2_operator_stop_count'] = seg2_stats['operator_stop_count']
            output_row['seg2_teammate_stop_count'] = seg2_stats['teammate_stop_count']
            output_row['seg2_supervisor_stop_count'] = seg2_stats['supervisor_stop_count']
            
            # Segment 3
            seg3_duration_s = (trial_end_ts - me2_ts) / 1e9
            seg3_stats = calculate_period_stats(me2_ts, trial_end_ts, mode_timeline_sorted)
            output_row['seg3_duration_s'] = seg3_duration_s
            output_row['seg3_operator_time_s'] = seg3_stats['operator_time_ns'] / 1e9
            output_row['seg3_pct_time_operator'] = (output_row['seg3_operator_time_s'] / seg3_duration_s) * 100 if seg3_duration_s > 0 else 0
            output_row['seg3_operator_stop_count'] = seg3_stats['operator_stop_count']
            output_row['seg3_teammate_stop_count'] = seg3_stats['teammate_stop_count']
            output_row['seg3_supervisor_stop_count'] = seg3_stats['supervisor_stop_count']
        else:
            event_rel_times = [event['trial-rel-time'] for event in marked_events]
            output_row['comment'] = f"TODO: Found {len(marked_events)} events at times: {event_rel_times}"
            
        final_results.append(output_row)
        
    # --- 4. Create and Save the Final DataFrame ---
    final_df = pd.DataFrame(final_results)
    
    # Define and set column order for better readability
    col_order = [
        'trial_number', 'comment', 
        'trial_duration_s', 'trial_operator_time_s', 'trial_pct_time_operator',
        'trial_operator_stop_count', 'trial_teammate_stop_count', 'trial_supervisor_stop_count',
        'seg1_duration_s', 'seg1_operator_time_s', 'seg1_pct_time_operator',
        'seg1_operator_stop_count', 'seg1_teammate_stop_count', 'seg1_supervisor_stop_count',
        'seg2_duration_s', 'seg2_operator_time_s', 'seg2_pct_time_operator',
        'seg2_operator_stop_count', 'seg2_teammate_stop_count', 'seg2_supervisor_stop_count',
        'seg3_duration_s', 'seg3_operator_time_s', 'seg3_pct_time_operator',
        'seg3_operator_stop_count', 'seg3_teammate_stop_count', 'seg3_supervisor_stop_count'
    ]
    
    # Reorder the DataFrame, adding columns that might be missing (e.g., for TODO trials)
    final_df = final_df.reindex(columns=col_order)
    
    # Save the final CSV
    final_df.to_csv(final_output_file, index=False, float_format='%.4f')
    print(f"\n--- Analysis Complete ---\nFinal summary saved to: {final_output_file}")

else:
    print("\nCould not proceed with Phase 3 because one or more required timeline files were not loaded.")

--- Starting Simplified Phase 3: Final Analysis ---
Successfully loaded event timeline from: ../../data/cohort-2-event-timeline.yaml
Successfully loaded mode timeline from: ../../data/cohort-2-mode-timeline.yaml

Processing trials...
 - Analyzing Trial 1...
 - Analyzing Trial 2...
 - Analyzing Trial 4...
 - Analyzing Trial 5...
 - Analyzing Trial 6...
 - Analyzing Trial 7...
 - Analyzing Trial 8...
 - Analyzing Trial 9...
 - Analyzing Trial 11...

--- Analysis Complete ---
Final summary saved to: ../../data/cohort-2-summary.csv


# The OTHER final analysis - dets and velocity

In [19]:
# --- Phase 4: Detection and Velocity Analysis ---

print("--- Starting Phase 4: Detection and Velocity Analysis ---")

# --- 1. Define Analysis Parameters ---
# These are the inputs you can configure for the analysis.
laser_min_range_m = 0.1  # Ranges below this value are considered erroneous and discarded.
laser_max_range_m = 10.0 # Used for area calculation.
laser_angle_rad = 3.14   # The field of view of the laser in radians.

# Calculate the sensor area once
laser_area = 0.5 * laser_angle_rad * (laser_max_range_m**2)
print(f"Calculated Laser Area: {laser_area:.4f} m^2")


# --- 2. Define the Core Analysis Function ---

def calculate_detection_velocity_stats(period_df: pd.DataFrame, min_range_threshold: float):
    """
    Calculates detection and velocity statistics for a given period (a slice of a DataFrame).
    
    Args:
        period_df: The slice of the DataFrame for the period to analyze.
        min_range_threshold: The minimum valid range to consider for detections.
        
    Returns:
        A dictionary of calculated statistics.
    """
    stats = {}
    
    # --- Minimum Detection Range ---
    all_ranges = []
    # Drop rows with no detection data, then iterate
    for json_str in period_df['detection_ranges_json'].dropna():
        try:
            # Filter ranges *as they are loaded*
            ranges_in_row = [r for r in json.loads(json_str) if r >= min_range_threshold]
            if ranges_in_row:
                all_ranges.extend(ranges_in_row)
        except (json.JSONDecodeError, TypeError):
            continue # Skip malformed JSON strings
            
    stats['min_detection_range'] = np.min(all_ranges) if all_ranges else np.nan
    stats['mean_detection_range'] = np.mean(all_ranges) if all_ranges else np.nan
    
    # --- Mean Detection Density ---
    # Drop rows where there was no detection message, but keep rows with 0 detections
    n_detections_series = period_df['n_detections'].dropna()
    if not n_detections_series.empty:
        mean_n_detections = n_detections_series.mean()
        stats['mean_detection_density'] = mean_n_detections / laser_area
        stats['mean_n_detections'] = mean_n_detections
    else:
        stats['mean_detection_density'] = np.nan
        stats['mean_n_detections'] = np.nan
        
    # --- Mean Velocity ---
    # Drop rows without velocity data and calculate the mean for each component
    vel_linear_series = period_df['vel_linear_x'].dropna()
    vel_ang_series = period_df['vel_ang_z'].dropna()
    
    stats['mean_vel_linear_x'] = vel_linear_series.mean() if not vel_linear_series.empty else np.nan
    stats['mean_vel_ang_z'] = vel_ang_series.mean() if not vel_ang_series.empty else np.nan
    
    return stats


# --- 3. Load Timelines, Process Trials, and Generate Report ---
try:
    with open(output_event_timeline_file, 'r') as f:
        event_timeline = yaml.safe_load(f)
    print(f"Successfully loaded event timeline from: {output_event_timeline_file}")
    
    # Load the original raw data CSV
    main_df = pd.read_csv(input_csv_file)
    print(f"Successfully loaded raw data from: {input_csv_file}")

    final_results = []
    print("\nProcessing trials for detection and velocity stats...")

    # Iterate through each trial defined in the event timeline
    for trial_num_str, trial_data in event_timeline.items():
        trial_num = int(trial_num_str)
        print(f" - Analyzing Trial {trial_num}...")
        
        # Get trial boundaries from the timeline
        trial_start_ts = trial_data['trial-start']['timestamp']
        trial_end_ts = trial_data['trial-end']['timestamp']

        output_row = {'trial_number': trial_num}
        
        # --- Calculate stats for the ENTIRE TRIAL ---
        trial_df_slice = main_df[(main_df['timestamp'] >= trial_start_ts) & (main_df['timestamp'] <= trial_end_ts)]
        trial_stats = calculate_detection_velocity_stats(trial_df_slice, laser_min_range_m)
        for key, val in trial_stats.items():
            output_row[f'trial_{key}'] = val

        # --- Calculate stats for SEGMENTS ---
        marked_events = [data for key, data in trial_data.items() if key.startswith('event-')]
        
        if len(marked_events) == 2:
            me1_ts = marked_events[0]['timestamp']
            me2_ts = marked_events[1]['timestamp']
            
            # Segment 1
            seg1_df_slice = trial_df_slice[trial_df_slice['timestamp'] <= me1_ts]
            seg1_stats = calculate_detection_velocity_stats(seg1_df_slice, laser_min_range_m)
            for key, val in seg1_stats.items(): output_row[f'seg1_{key}'] = val
            
            # Segment 2
            seg2_df_slice = trial_df_slice[(trial_df_slice['timestamp'] > me1_ts) & (trial_df_slice['timestamp'] <= me2_ts)]
            seg2_stats = calculate_detection_velocity_stats(seg2_df_slice, laser_min_range_m)
            for key, val in seg2_stats.items(): output_row[f'seg2_{key}'] = val

            # Segment 3
            seg3_df_slice = trial_df_slice[trial_df_slice['timestamp'] > me2_ts]
            seg3_stats = calculate_detection_velocity_stats(seg3_df_slice, laser_min_range_m)
            for key, val in seg3_stats.items(): output_row[f'seg3_{key}'] = val
        else:
            # If segments aren't ideal, fill segment columns with NaN
            for i in range(1, 4):
                for stat_name in ['min_detection_range', 'mean_detection_range', 'mean_n_detections','mean_detection_density', 'mean_vel_linear_x', 'mean_vel_ang_z']:
                     output_row[f'seg{i}_{stat_name}'] = np.nan
            
        final_results.append(output_row)
        
    # --- 4. Create and Save the Final DataFrame ---
    final_df = pd.DataFrame(final_results)
    
    # Define and set column order for better readability
    col_order = ['trial_number']
    stat_cols = ['min_detection_range','mean_detection_range', 'mean_n_detections', 'mean_detection_density', 'mean_vel_linear_x', 'mean_vel_ang_z']
    for period in ['trial', 'seg1', 'seg2', 'seg3']:
        for col in stat_cols:
            col_order.append(f"{period}_{col}")
            
    # Reorder the DataFrame, ensuring all columns are present
    final_df = final_df.reindex(columns=col_order)
    
    # Save the final CSV
    final_output_file = Path(f'../../data/cohort-{cohort_number}-summary-det-vel.csv')
    final_df.to_csv(final_output_file, index=False, float_format='%.4f')
    print(f"\n--- Analysis Complete ---\nDetection and velocity summary saved to: {final_output_file}")

except (FileNotFoundError, NameError) as e:
    print(f"\nCould not execute Phase 4. Please ensure the previous phases have been run and the required files exist.")
    print(f"Error details: {e}")

--- Starting Phase 4: Detection and Velocity Analysis ---
Calculated Laser Area: 157.0000 m^2
Successfully loaded event timeline from: ../../data/cohort-4-event-timeline.yaml
Successfully loaded raw data from: ../../data/cohort-4-raw.csv

Processing trials for detection and velocity stats...
 - Analyzing Trial 1...
 - Analyzing Trial 2...
 - Analyzing Trial 3...
 - Analyzing Trial 4...
 - Analyzing Trial 5...
 - Analyzing Trial 6...
 - Analyzing Trial 7...
 - Analyzing Trial 8...
 - Analyzing Trial 9...
 - Analyzing Trial 10...
 - Analyzing Trial 11...
 - Analyzing Trial 12...

--- Analysis Complete ---
Detection and velocity summary saved to: ../../data/cohort-4-summary-det-vel.csv
